# CatBoost with binary keyword features (text only vs. keywords only vs. both)

In [ ]:
DATA_DIR = "../data"

In [ ]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm

from transformers import (
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    AutoModelForSequenceClassification,
    EarlyStoppingCallback
)

import tensorflow as tf
from catboost import CatBoostClassifier, Pool

from sklearn.model_selection import train_test_split, cross_val_score, RepeatedStratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, recall_score, precision_score, \
    balanced_accuracy_score
from datasets import Dataset, DatasetDict

In [ ]:
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
train_df = pd.read_csv(f'{DATA_DIR}/train.csv', sep=";")
test_df = pd.read_csv(f'{DATA_DIR}/test.csv', sep=";")
train_df.head()

,text,label
0,Kunjungan Prabowo ini untuk meresmikan dan men...,Sumber Daya Alam
1,RT Anies dapat tepuk tangan meriah saat jadi R...,Politik
2,@CIqXqwGAT04tMtx4OCATxjoVq7vv/Y8HeYaIOgMFg8Y= ...,Demografi
3,RT @L3R8XFBw3WGbxRPSj0/0hHZTbqVGX7qtfwRg9zmhK7...,Politik
4,Anies Baswedan Harap ASN termasuk TNI dan Polr...,Politik


In [ ]:
def remove_noise(text):
    text = str(text)
    # remove mentions
    text = re.sub(r'@[\S+]+', '', text)
    # remove hashtags
    # text = re.sub(r'#\w+', '', text)
    # remove RT
    text = re.sub(r'RT[\s]+', '', text)
    # remove [RE ...]
    text = re.sub(r'\[RE\s+(\w+)\]', r'\1', text)
    # remove urls
    text = re.sub(r'https?:\/\/\S+', '', text)
    # remove emojis
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    # fix &amp;
    text = re.sub(r'&amp;', 'dan', text)
    # fix w/
    text = re.sub(r'w/', 'with', text)
    # fix 3
    text = re.sub(r'(?<=\w)3(?=\w)', 'e', text)
    # remove any non alphanumeric characters
    text = re.sub(r'[^A-Za-z0-9 ]', ' ', text)
    # fix unspaced words
    new_text = []
    for word in text.split():
        # weird word
        if word in ["BEiqnH5xQr", "JN18A"]:
            continue
        # fix shorten names
        elif word == "JK":
            new_text.append("jusuf kalla")
        elif word.lower() == "jkw":
            new_text.append("jokowi")
        elif word.lower() in ["bowo", "ayahbowo"]:
            new_text.append("prabowo")
        elif word.lower() in ["ayahbowogbran", "pbowogbran", "masbowogbran"]:
            new_text.append("pabrowo gibran")
        elif word == "MasGBRAN":
            new_text.append("gibran")
        elif word == "SBY":
            new_text.append("susilo bambang yudhoyono")
        elif word == "AHY":
            new_text.append("agnes haryani")
        # keep as is
        elif re.fullmatch(r'[A-Z]+', word) or word.lower() in ["s1", "s2", "s3"]:
            new_text.append(word)
        else:
            # fix unspaced words
            new_text.append(re.sub(r'(?<=[a-z])(?=[A-Z])|(?<=[A-Za-z])(?=[0-9])|(?<=[0-9])(?=[A-Za-z])|(?<=[A-Z])(?=[A-Z][a-z])', ' ', word))
    text = ' '.join(new_text)
    # remove extra spaces
    text = re.sub(r'\s+', ' ', text)
    # remove whitespace leading & trailing
    text = text.strip()
    return text

def normalize_word(text):
    map = {
        "sj": "saja",
        "yg": "yang",
        "dgn": "dengan",
        "dg": "dengan",
        "dpt": "dapat",
        "dptkan": "dapatkan",
        "hnya": "hanya",
        "skrng": "sekarang",
        "uu": "undang undang",
        "asn": "aparatur sipil negara",
        "ordal": "orang dalam",
        "mrk": "mereka",
        "rmh": "rumah",
        "kpk": "komisi pemberantasan korupsi",
        "blg": "bilang",
        "tuch": "tuh",
        "umkm": "usaha mikro kecil menengah",
        "jis": "jakarta international stadium",
        "jkt": "jakarta",
        "spt": "seperti",
        "tps": "tempat pemungutan suara",
        "alkes": "alat kesehatan",
        "bowo": "prabowo",
    }

    text = text.replace("UU ITE", "Undang Undang Informasi dan Transaksi Elektronik")
    text = text.replace("kitamasbowo dangbran", "kita pilih prabowo dan gibran")
    text = text.replace("PDI P", "PDIP")
    text = text.replace("PDI Perjuangan", "PDIP")

    party_names = [
        "pkb",
        "gerindra",
        "pdi",
        "pdip",
        "golkar",
        "nasdem",
        "gelora",
        "pks",
        "pkn",
        "hanura",
        "pan",
        "pbb",
        "demokrat",
        "psi",
        "perindo",
        "ppp",
        "ummat",
        "pna",
        "gabthat",
        "pda",
        "sira"
    ]

    stop_words = [
        "dekade08",
        "tempodotco",
        "md"
    ]

    new_text = []
    for word in text.split():
        if word.lower() in map:
            new_text.append(map[word.lower()])
        elif word.lower() in party_names:
            new_text.append("Partai Politik")
        elif word.lower() in stop_words:
            continue
        else:
            new_text.append(word)

    text = ' '.join(new_text)
    return text

def drop_duplicated(df):
    print("Df size:", df.shape)
    df.drop(df[df['text'].str.len() == 0].index, inplace=True)
    print("After dropping empty text rows, df size: ", df.shape)
    df.drop_duplicates(subset=['text', 'label'], keep='first', inplace=True)
    print("After dropping literal duplicated rows, df size: ", df.shape)
    df.drop_duplicates(subset=['text'], keep=False, inplace=True)
    print("After dropping text duplicated rows, df size: ", df.shape)
    return df

In [ ]:
train_df['text'] = train_df['text'].apply(remove_noise)
train_df['text'] = train_df['text'].apply(normalize_word)
train_df = drop_duplicated(train_df)
train_df.head()

Df size: (5000, 2)
After dropping empty text rows, df size:  (4991, 2)
After dropping literal duplicated rows, df size:  (4353, 2)
After dropping text duplicated rows, df size:  (4240, 2)


,text,label
0,Kunjungan Prabowo ini untuk meresmikan dan men...,Sumber Daya Alam
1,Anies dapat tepuk tangan meriah saat jadi Rekt...,Politik
2,emng bener sih pendukung 01 ada yang goblok be...,Demografi
3,Sewaktu anies bersikap kritis ke kinerja pak p...,Politik
4,Anies Baswedan Harap aparatur sipil negara ter...,Politik


In [ ]:
train_df.to_csv(f"{DATA_DIR}/train_cleaned_v2.csv", index=False)

In [ ]:
train_df.label.value_counts()

label
Politik                    2867
Sosial Budaya               384
Ekonomi                     264
Pertahanan dan Keamanan     259
Ideologi                    255
Sumber Daya Alam            133
Demografi                    55
Geografi                     19
Name: count, dtype: int64

In [ ]:
test_df['Text'] = test_df['Text'].apply(remove_noise)
test_df['Text'] = test_df['Text'].apply(normalize_word)
test_df.head()

,IDText,Text
0,TXT0001,Lu mau org 2 pro demokrasi di negara ini bisa ...
1,TXT0002,Prabowo ditanya soal hutang luar negeri dia me...
2,TXT0003,kiki daliyo Ganjar Pranowo itulah beliau sosok...
3,TXT0004,Prabowo Gibran yang bisa melakukan itu semua d...
4,TXT0005,Lah justru yang gak nyambung junjungan elu aom...


In [ ]:
train_df[train_df['text'].isin(test_df['Text'])].drop_duplicates(subset=['text'])

,text,label
64,Anies itu prestasinya di Jakarta banyak sekali...,Politik
68,Isu Palestina tidak penting bagi Prabowo tidak...,Politik
87,Palestina di tindas karena gk punya kekuatan m...,Pertahanan dan Keamanan
160,Partai Politik Tanggapi Jokowi Prabowo Makan M...,Politik
270,nahh itu dia laeku ga bisa dibuka semua data i...,Politik
...,...,...
4869,Jadi ikut sedih juga Pak Prabowo memang sering...,Politik
4881,saat ini warga kampung bayam seharusnya bisa h...,Politik
4946,Percayalah orang pertama yang bakal didepak Pr...,Politik
4953,Anies kehilangan Otoritas bicara etika pada de...,Politik


In [ ]:
kw_df = pd.read_excel(f"{DATA_DIR}/keywords_prob.xlsx")
kw_df.head()

,keyword,Demografi,Ekonomi,Geografi,Ideologi,Pertahanan dan Keamanan,Sosial Budaya,Sumber Daya Alam,Politik
0,pendidikan rendah,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,kelompok etnis,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,masyarakat lintas etnis,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,kaum difabel,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,bugis,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
train_kw_df = train_df.copy()
train_kw_df.head()

,text,label
0,Kunjungan Prabowo ini untuk meresmikan dan men...,Sumber Daya Alam
1,Anies dapat tepuk tangan meriah saat jadi Rekt...,Politik
2,emng bener sih pendukung 01 ada yang goblok be...,Demografi
3,Sewaktu anies bersikap kritis ke kinerja pak p...,Politik
4,Anies Baswedan Harap aparatur sipil negara ter...,Politik


In [ ]:
for kw in kw_df['keyword']:
    train_kw_df[kw] = train_kw_df['text'].str.lower().str.contains(kw).astype(int)

train_kw_df.head()

,text,label,pendidikan rendah,kelompok etnis,masyarakat lintas etnis,kaum difabel,bugis,generasi emas,low educated,menengah bawah,...,nyerocos,manyargubi,politik merdeka,futrijasun,vidaanasution,imadarch,chairudinn6548,sering ngibul,tata kata,ryan_fighter7
0,Kunjungan Prabowo ini untuk meresmikan dan men...,Sumber Daya Alam,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Anies dapat tepuk tangan meriah saat jadi Rekt...,Politik,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,emng bener sih pendukung 01 ada yang goblok be...,Demografi,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Sewaktu anies bersikap kritis ke kinerja pak p...,Politik,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,Anies Baswedan Harap aparatur sipil negara ter...,Politik,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# give me rows where all the columns is 0
train_kw_df[(train_kw_df[kw_df['keyword']] == 0).all(axis=1)].label.value_counts()

label
Politik                    53
Sumber Daya Alam            5
Sosial Budaya               4
Ideologi                    3
Pertahanan dan Keamanan     2
Ekonomi                     2
Geografi                    1
Name: count, dtype: int64

In [ ]:
# drop rows where all the columns is 0
train_kw_df.drop(train_kw_df[(train_kw_df[kw_df['keyword']] == 0).all(axis=1)].index, inplace=True)
train_kw_df.shape

(4170, 1766)

In [ ]:
train_kw_df.label.value_counts()

label
Politik                    2816
Sosial Budaya               380
Ekonomi                     264
Pertahanan dan Keamanan     257
Ideologi                    252
Sumber Daya Alam            128
Demografi                    55
Geografi                     18
Name: count, dtype: int64

In [ ]:
# Split the DataFrame into training and validation sets
train_df_split, val_df_split = train_test_split(train_kw_df, test_size=0.2, stratify=train_kw_df['label'], random_state=42)

In [ ]:
train_df_split.reset_index(drop=True, inplace=True)
val_df_split.reset_index(drop=True, inplace=True)

train_pool = Pool(
    data=train_df_split[kw_df["keyword"].to_list() + ["text"]].to_numpy(),
    label=train_df_split['label'].to_numpy(),
    text_features=['text'],
    feature_names=kw_df["keyword"].to_list() + ["text"]
)

val_pool = Pool(
    data=val_df_split[kw_df["keyword"].to_list() + ["text"]].to_numpy(),
    label=val_df_split['label'].to_numpy(),
    text_features=['text'],
    feature_names=kw_df["keyword"].to_list() + ["text"]
)

In [ ]:
train_df_split["label"].value_counts()

label
Politik                    2253
Sosial Budaya               304
Ekonomi                     211
Pertahanan dan Keamanan     206
Ideologi                    202
Sumber Daya Alam            102
Demografi                    44
Geografi                     14
Name: count, dtype: int64

In [ ]:
class_weights_dict = dict()
n, num_class = train_df_split.shape[0], len(train_df_split['label'].unique())
for label in train_df_split['label'].unique():
    n_l = train_df_split[train_df_split['label'] == label].shape[0]
    class_weights_dict[label] = n / (num_class * n_l)

class_weights_dict

{'Politik': 0.18508655126498003,
 'Ideologi': 2.0643564356435644,
 'Sumber Daya Alam': 4.088235294117647,
 'Ekonomi': 1.9763033175355451,
 'Pertahanan dan Keamanan': 2.0242718446601944,
 'Sosial Budaya': 1.3717105263157894,
 'Geografi': 29.785714285714285,
 'Demografi': 9.477272727272727}

In [ ]:
cb_model = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='TotalF1',
    task_type="GPU",
    devices="0",
    verbose=100,
    random_state=42,
    class_weights = class_weights_dict
)

cb_model.fit(
    train_pool,
    eval_set=val_pool
)

Learning rate set to 0.118762
0:	learn: 0.4010800	test: 0.3385659	best: 0.3385659 (0)	total: 13.4ms	remaining: 13.4s
100:	learn: 0.7383314	test: 0.4717561	best: 0.4849368 (72)	total: 1.17s	remaining: 10.4s
200:	learn: 0.7948872	test: 0.4781996	best: 0.4849368 (72)	total: 2.29s	remaining: 9.11s
300:	learn: 0.8237006	test: 0.4879576	best: 0.4879576 (300)	total: 3.4s	remaining: 7.89s
400:	learn: 0.8420501	test: 0.4960602	best: 0.4960602 (400)	total: 4.61s	remaining: 6.88s
500:	learn: 0.8647006	test: 0.4820779	best: 0.4960722 (401)	total: 5.73s	remaining: 5.7s
600:	learn: 0.8833249	test: 0.4834901	best: 0.4960722 (401)	total: 6.61s	remaining: 4.39s
700:	learn: 0.9027454	test: 0.4895669	best: 0.4960722 (401)	total: 7.5s	remaining: 3.2s
800:	learn: 0.9170146	test: 0.4773085	best: 0.4960722 (401)	total: 8.41s	remaining: 2.09s
900:	learn: 0.9288265	test: 0.4743722	best: 0.4960722 (401)	total: 9.31s	remaining: 1.02s
999:	learn: 0.9394505	test: 0.4809444	best: 0.4960722 (401)	total: 10.2s	remain

In [ ]:
# Only Text
# Make predictions on the validation set
predictions = cb_model.predict(val_pool)

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(val_df_split['label'], predictions)
print("Balanced Accuracy:", balanced_acc)

Balanced Accuracy: 0.4722105392058831


In [ ]:
# Only Binary
# Make predictions on the validation set
predictions = cb_model.predict(val_pool)

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(val_df_split['label'], predictions)
print("Balanced Accuracy:", balanced_acc)

Balanced Accuracy: 0.47071751466875755


In [ ]:
# Text + Binary
# Make predictions on the validation set
predictions = cb_model.predict(val_pool)

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(val_df_split['label'], predictions)
print("Balanced Accuracy:", balanced_acc)

Balanced Accuracy: 0.5126369956374881


In [ ]:
model_1 = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='TotalF1',
    task_type="GPU",
    devices="0",
    verbose=0,
    random_state=42,
    class_weights=class_weights_dict,
    text_features=['text']
)


cv_method = RepeatedStratifiedKFold(n_splits=5, n_repeats=1, random_state=42)
cv_results_model_1 = cross_val_score(estimator=model_1, X=train_kw_df.drop("label", axis=1), y=train_kw_df["label"], cv=cv_method,
                                     verbose=100, scoring='balanced_accuracy')

print("Model 1 - Balanced Accuracy scores: ")
print(cv_results_model_1)
print("Mean: ")
print(cv_results_model_1.mean().round(3))

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[CV] START .....................................................................
[CV] END ................................ score: (test=0.542) total time=  10.7s
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:   10.7s remaining:    0.0s
[CV] START .....................................................................
[CV] END ................................ score: (test=0.539) total time=   9.0s
[Parallel(n_jobs=1)]: Done   2 out of   2 | elapsed:   19.8s remaining:    0.0s
[CV] START .....................................................................
[CV] END ................................ score: (test=0.470) total time=   9.5s
[Parallel(n_jobs=1)]: Done   3 out of   3 | elapsed:   29.4s remaining:    0.0s
[CV] START .....................................................................
[CV] END ................................ score: (test=0.470) total time=   9.1s
[Parallel(n_jobs=1)]: Done   4 